In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)


def chat():
    print("Iris (AI): Hi! Ask me anything (type 'exit' to quit)\n")

    history = None

    while True:
        text = input("You: ")

        if text.lower() in ["exit", "quit"]:
            print("Iris (AI): Ending chat. Bye!")
            break

        if text.strip() == "":
            print("Iris (AI): Please say something 🙂")
            continue

        new_input_ids = tokenizer.encode(text + tokenizer.eos_token, return_tensors="pt")

        # Limit history size to avoid overflow
        if history is not None:
            bot_input_ids = torch.cat([history, new_input_ids], dim=-1)
        else:
            bot_input_ids = new_input_ids

        output_ids = model.generate(
            bot_input_ids,
            max_length=300,  # reduced for better speed
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.9,
            top_k=50,
            top_p=0.95,
            no_repeat_ngram_size=2  # prevents repetition
        )

        # Keep only recent tokens (important fix)
        history = output_ids[:, -500:]

        response = tokenizer.decode(
            output_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )
        print(f"\nYou: {text}")
        print(f"Iris (AI): {response}\n") #Name of AI Chatbot


if __name__ == "__main__":
    chat()

C:\Users\adesh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Iris (AI): Hi! Ask me anything (type 'exit' to quit)



The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



You: Hello
Iris (AI): Hey, I am interested in the deal if you are still interested.


You: How are you?
Iris (AI): I am good, how about you?


You: I am good too
Iris (AI): Ok I'll go in. What's the price for the items? EDIT : Nvm, someone else helped. Thanks though

Iris (AI): Please say something 🙂
Iris (AI): Ending chat. Bye!
